In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax/소득세법_with_markdown.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=20
)

documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 3.09 s
Wall time: 3.33 s


180

In [3]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
    model = 'embedding-query'
)

In [4]:
len(embedding.embed_query("소득세법"))

4096

In [5]:
from pinecone import Pinecone
pc = Pinecone()
[index.name for index in pc.list_indexes()]

['better-rag-index',
 'tax-index-openai',
 'tax-index-markdown',
 'tax-index',
 'tax-index-table']

In [6]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec

# pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 인덱스 설정
index_name = 'tax-index-markdown'
dimension = len(embedding.embed_query("차원 확인용"))

# 인덱스 생성 (없을 경우에만 생성)
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print(f'인덱스 생성 : {index_name}')
else:
    print(f'중복된 인덱스 : {index_name}')

중복된 인덱스 : tax-index-markdown


In [7]:
# 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['better-rag-index', 'tax-index-openai', 'tax-index-markdown', 'tax-index', 'tax-index-table']
True


C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 180}},
 'total_vector_count': 180,
 'vector_type': 'dense'}


In [8]:
# %%time
# # 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# from pinecone import Pinecone
# pc = Pinecone()
# from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

In [9]:
# 업로드한 벡터DB를 load할 때
from pinecone import Pinecone
pc = Pinecone()
from langchain_pinecone import PineconeVectorStore
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 2. 답변 생성 전 retrieval 확인

In [10]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=2) # 기본k값은 4

In [11]:
retriever = database.as_retriever( 
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")

In [12]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])

In [13]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [14]:
from openai import OpenAI
client = OpenAI()
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""- [context]를 참고해서 사용자의 질문에 답변해 주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [15]:
result = response.choices[0].message.content

In [16]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(
    model = 'solar-pro4',
    reasoning_effort='low'
)

In [17]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- 다음의 [context]를 참고하여 사용자의 질문에 답변해 주세요.
{retrieved_doc}
- 질문 : {query}"""

In [18]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

제공해주신 [context]의 소득세법 조문(제55조 세율, 제59조 근로소득세액공제 등)을 바탕으로 연봉 5천만원인 직장인의 소득세 계산 방법과 산출 구조를 설명해 드리겠습니다.

정확한 소득세액은 연봉(총급여액)에서 각종 소득공제(본인 기본공제, 연금보험료, 보험료, 신용카드 사용액 등)를 차감한 **종합소득과세표준**을 기준으로 계산되며, 최종적으로는 근로소득세액공제 등을 적용하여 확정됩니다. 제공된 지문을 기준으로 계산 과정을 단계별로 설명해 드립니다.

### 1. 종합소득산출세액 계산 (제55조 세율 적용)

소득세는 연봉 전체가 아니라, 연봉에서 소득공제 등을 뺀 **과세표준**에 세율을 적용합니다. 만약 다른 공제가 없이 과세표준이 연봉과 동일하게 5,000만원이라고 가정할 경우, 제55조 제1항의 세율 테이블을 적용합니다.

*   **해당 구간:** 1,400만원 초과 5,000만원 이하
*   **세율 적용식:** 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)
*   **계산:**
    *   초과금액: 5,000만원 - 1,400만원 = 3,600만원
    *   초과금액 세액: 3,600만원 × 15% = 540만원
    *   **종합소득산출세액 (가정):** 84만원 + 540만원 = **624만원**

※ 실제로 근로소득자는 인적공제, 특별소득공제, 그 밖의 소득공제 등을 적용받으므로 과세표준은 5,000만원보다 낮아지며, 이에 따라 세액도 줄어듭니다.

### 2. 근로소득세액공제 계산 (제59조 적용)

근로소득이 있는 거주자는 제59조에 따라 계산된 종합소득산출세액에서 일정 금액을 공제받을 수 있습니다. 질문의 연봉(총급여액) 5,000만원을 기준으로 제59조 제2항의 한도 규정을 적용합니다.

*   **총급여액 구간:** 3,300만원 초과 7,000만원 이하 (제2항 제2호)
*   **공제액 산정 공식:** 74만원 - [(총급여액 - 3,300만원) × 8/1,000]
*   **계산:**
    *

# LangChain

In [19]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# vector DB load
# pc = Pinecone()
index_name = 'tax-index-markdown'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [20]:
# RAG 렝체인 구성 (LCEL방식)
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥의 세율은 과세표준 기준이며, 근로소득공제 등 각종 공제 적용 전이므로 연봉 5천만원만으로는 정확한 소득세를 계산할 수 없습니다. 기본 공제 없이 해당 세율을 단순 적용하면 과세표준이 5천만원 이하일 때 84만원+(5천만원-1,400만원)×15% 공식에 따라 약 559만원으로 산출되나, 이는 실제 납부세액과 다릅니다. 정확한 세액은 근로소득공제, 인적공제, 자녀세액공제, 연금계좌세액공제 등 개인별 공제 항목에 따라 달라지므로 단정할 수 없습니다.


# 4. 키워드 사전 활용

In [21]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
# query = '종합소득이 5,000만원인 거주자의 소득세는 얼마인가요?'

In [ ]:
import numpy as np
def cosine_similarity(vec1, vec2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)
vec1 = embedding.embed_query("직장인")
vec2 = embedding.embed_query("거주자")
cosine_similarity(vec1, vec2)

In [ ]:
vec1 = embedding.embed_query("연봉")
vec2 = embedding.embed_query("종합소득")
cosine_similarity(vec1, vec2)

In [ ]:
vec1 = embedding.embed_query("5천")
vec2 = embedding.embed_query("5,000")
cosine_similarity(vec1, vec2)

In [ ]:
# 키워드 사전 : 일반사용자 단어 -> 소득세법 용어
from langchain_core.prompts import ChatPromptTemplate
dictionary = ["연봉->종합소득", "사람을 나타내는 표현->거주자", "5천->5,000"]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요
- 사전 : {dictionary}
- 질문 : {{question}}""")

In [ ]:
strOutputParser = StrOutputParser()
strOutputParser.invoke(llm.invoke(prompt.invoke({"question":"소득이 높은 남자가 있습니다"})))

In [ ]:
# langchain
dictionary_chain = prompt | llm | StrOutputParser()
#dictionary_chain.invoke({"question":"연봉이 5천만원인 직장인은?"})
dictionary_chain.invoke("연봉이 5천만원인 직장인은?")

In [ ]:
new_chain = dictionary_chain | rag_chain

In [ ]:
new_chain.invoke("연봉이 5천만원인 직장인의 소득세는 얼마예요?")
# new_chain.invoke({"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"})

```
입력 {"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"}
       ↓
dictionary_chain에 의해 질문 수정(documents용어 수정)
       ↓
  "개선된 질문"
       ↓
  retriever
       ↓ 
format_documents
       ↓ 
rag_chain의 prompt + llm
      ↓ 
strOutputParser()
```